In [21]:
! pip install openai


In [22]:
import os
from openai import OpenAI

In [23]:
from openai import OpenAI
from google.colab import userdata

# Get the Groq API key from Colab's secrets manager
GROQ_API_KEY = userdata.get('Groq_API_Key')

# Ensure the API key is not None
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab secrets. Please add it.")

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

response = client.responses.create(
    input="Explain the importance of fast language models",
    model="openai/gpt-oss-20b",
)
print(response.output_text)

# The “Fast” Advantage: Why Speed Matters for Language Models

Fast language models—those that can answer a prompt in milliseconds and generate text at a few hundred tokens per second—are not a luxury; they’re a prerequisite for many of the real‑world applications that depend on large‑scale NLP today.  
Below we unpack the key reasons why speed matters, the practical implications for developers and organizations, and the techniques that make it possible.

---

## 1. User Experience & Human‑Computer Interaction

| Application | Latency Tolerance | Typical Speed Required |
|-------------|-------------------|------------------------|
| Chatbots / Customer support | < 200 ms | 200‑800 tokens/s |
| Voice assistants (e.g., Alexa, Siri) | < 300 ms | 300‑1,000 tokens/s |
| Real‑time translation | < 500 ms | 400‑1,500 tokens/s |
| Code autocompletion | < 100 ms | 500‑2,000 tokens/s |
| Interactive storytelling / games | < 200 ms | 300‑800 tokens/s |

When an answer takes seconds to materialize,

In [24]:
! pip install langchain faiss-cpu langchain_community tiktoken sentence-transformers

In [25]:
#step 1 load document
from langchain_community.document_loaders import TextLoader
loader=TextLoader('/content/placement_assistant.txt')
documents=loader.load()

In [26]:
# step 2 create the embedding + vector DB
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the HuggingFace embeddings
# Using a common open-source model; ensure 'sentence-transformers' is installed
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# Create the vector database from your pre-loaded documents
vector_db = FAISS.from_documents(documents, embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [27]:
# #step 3 Retrieval + Generation
def rag_query(query):
    # Retrieve the top 3 most relevant document chunks based on the query
    docs = vector_db.similarity_search(query, k=3)

    # Combine the content of the retrieved documents into a single context string
    context = " ".join([doc.page_content for doc in docs])

    # Generate the response using OpenAI's chat completions API
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", # Updated model to the user-specified Groq model
        messages=[
            {"role": "system", "content": "use the provided context to answer the question"},
            {"role": "user", "content": f"context: {context}\n\nquery: {query}"}
        ]
    )

    return response.choices[0].message.content

In [28]:
print(rag_query("What is Object Oriented Programming?"))

OOP is a programming paradigm based on classes and objects.


In [29]:
import warnings
warnings.filterwarnings('ignore')
!pip install bitsandbytes accelerate transformers


In [30]:
# #step 1 : Load model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import bitsandbytes as bnb

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# #define the bitsandbytes config for 8 bit quantization
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [31]:
# #step 2 : Apply LoRA(PEFT)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# #preapre the model for k bit trining
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.5,
    bias="none",
    task_type="CAUSAL_LM"
)
model=get_peft_model(model,lora_config)


In [41]:
from datasets import Dataset

data = [
    {"text": "Q: What is Data Structures and Algorithms?\nA: Data Structures and Algorithms are techniques used to organize and process data efficiently."},

    {"text": "Q: Why is DSA important for placements?\nA: DSA helps solve coding problems efficiently and is commonly asked in interviews."},

    {"text": "Q: What is an array?\nA: An array is a collection of elements stored in contiguous memory locations."},

    {"text": "Q: What is a linked list?\nA: A linked list is a linear data structure where elements are connected using pointers."},

    {"text": "Q: What is a stack?\nA: A stack is a data structure that follows the Last In First Out principle."},

    {"text": "Q: What is a queue?\nA: A queue is a data structure that follows the First In First Out principle."},

    {"text": "Q: What is recursion?\nA: Recursion is a programming technique where a function calls itself."},

    {"text": "Q: What is time complexity?\nA: Time complexity measures the execution time of an algorithm."},

    {"text": "Q: What is space complexity?\nA: Space complexity measures the memory used by an algorithm."},

    {"text": "Q: What is Big O notation?\nA: Big O notation describes algorithm efficiency."},

    {"text": "Q: What is Object Oriented Programming?\nA: OOP is a programming paradigm based on classes and objects."},

    {"text": "Q: What are the four pillars of OOP?\nA: Encapsulation, Inheritance, Polymorphism and Abstraction."},

    {"text": "Q: What is encapsulation?\nA: Encapsulation is wrapping data and methods together inside a class."},

    {"text": "Q: What is inheritance?\nA: Inheritance allows one class to acquire properties of another class."},

    {"text": "Q: What is polymorphism?\nA: Polymorphism allows methods to behave differently based on the object."},

    {"text": "Q: What is abstraction?\nA: Abstraction hides implementation details and shows only essential features."},

    {"text": "Q: What is DBMS?\nA: DBMS is software used to manage and organize databases."},

    {"text": "Q: What is SQL?\nA: SQL is a language used to interact with relational databases."},

    {"text": "Q: What is a primary key?\nA: A primary key uniquely identifies a record in a table."},

    {"text": "Q: What is a foreign key?\nA: A foreign key creates a relationship between two tables."},

    {"text": "Q: What is normalization?\nA: Normalization reduces redundancy and improves data consistency."},

    {"text": "Q: What is a join?\nA: A join combines data from multiple tables."},

    {"text": "Q: What is an operating system?\nA: An operating system manages hardware and software resources."},

    {"text": "Q: What is a process?\nA: A process is a program currently in execution."},

    {"text": "Q: What is a thread?\nA: A thread is the smallest unit of execution within a process."},

    {"text": "Q: What is deadlock?\nA: Deadlock occurs when processes wait indefinitely for resources."},

    {"text": "Q: What is CPU scheduling?\nA: CPU scheduling determines which process gets CPU time."},

    {"text": "Q: What is computer networking?\nA: Networking connects devices to exchange information."},

    {"text": "Q: What is an IP address?\nA: An IP address uniquely identifies a device on a network."},

    {"text": "Q: What is DNS?\nA: DNS translates domain names into IP addresses."},

    {"text": "Q: What is HTTP?\nA: HTTP is a protocol used for communication between clients and servers."},

    {"text": "Q: What is HTTPS?\nA: HTTPS is the secure version of HTTP."},

    {"text": "Q: What is cloud computing?\nA: Cloud computing provides computing resources over the internet."},

    {"text": "Q: What is AWS?\nA: AWS is Amazon Web Services, a cloud computing platform."},

    {"text": "Q: What is Amazon EC2?\nA: Amazon EC2 provides scalable virtual servers in the cloud."},

    {"text": "Q: What is Amazon S3?\nA: Amazon S3 is a cloud storage service."},

    {"text": "Q: What is Git?\nA: Git is a distributed version control system."},

    {"text": "Q: What is GitHub?\nA: GitHub is a platform for hosting Git repositories."},

    {"text": "Q: What is a repository?\nA: A repository stores project files and version history."},

    {"text": "Q: What is branching in Git?\nA: Branching allows parallel development of features."},

    {"text": "Q: What is merge?\nA: Merge combines changes from different branches."},

    {"text": "Q: What is a resume?\nA: A resume summarizes education, skills and experience."},

    {"text": "Q: What should a fresher resume contain?\nA: Education, skills, projects, certifications and achievements."},

    {"text": "Q: What is an internship?\nA: An internship provides practical industry experience."},

    {"text": "Q: What is a technical interview?\nA: A technical interview evaluates coding and technical knowledge."},

    {"text": "Q: What is an HR interview?\nA: An HR interview evaluates communication and personality."},

    {"text": "Q: How should I introduce myself?\nA: Introduce your education, skills, projects and career goals."},

    {"text": "Q: Why should we hire you?\nA: Because I possess relevant skills and a strong willingness to learn."},

    {"text": "Q: What are your strengths?\nA: Problem solving, teamwork, adaptability and communication."},

    {"text": "Q: What is aptitude?\nA: Aptitude measures logical and analytical thinking ability."},
    {"text":"Q: What is percentage?\nA: Percentage is a number expressed as a fraction of 100."},
{"text":"Q: What is profit?\nA: Profit is the amount gained when selling price exceeds cost price."},
{"text":"Q: What is loss?\nA: Loss occurs when cost price exceeds selling price."},
{"text":"Q: What is simple interest?\nA: Simple interest is calculated only on the principal amount."},
{"text":"Q: What is compound interest?\nA: Compound interest is calculated on both principal and accumulated interest."},
{"text":"Q: What is ratio?\nA: Ratio compares two quantities."},
{"text":"Q: What is proportion?\nA: Proportion states that two ratios are equal."},
{"text":"Q: What is probability?\nA: Probability measures the likelihood of an event occurring."},
{"text":"Q: What is permutation?\nA: Permutation is an arrangement of objects in a specific order."},
{"text":"Q: What is combination?\nA: Combination is a selection of objects regardless of order."},
    {"text":"Q: Tell me about yourself.\nA: I am a motivated student with strong technical and problem-solving skills."},
{"text":"Q: Why do you want this job?\nA: This role aligns with my skills and career goals."},
{"text":"Q: What are your strengths?\nA: My strengths include adaptability, teamwork, and problem solving."},
{"text":"Q: What are your weaknesses?\nA: I sometimes spend extra time perfecting tasks, but I manage it effectively."},
{"text":"Q: Where do you see yourself in five years?\nA: I see myself growing into a skilled professional with leadership responsibilities."},
{"text":"Q: Why should we hire you?\nA: I have the required skills, enthusiasm, and willingness to learn."},
{"text":"Q: What motivates you?\nA: Continuous learning and achieving meaningful goals motivate me."},
{"text":"Q: Are you a team player?\nA: Yes, I collaborate effectively and contribute positively to team success."},
{"text":"Q: How do you handle pressure?\nA: I stay organized, prioritize tasks, and remain focused."},
{"text":"Q: What are your career goals?\nA: My goal is to become a skilled software engineer and contribute to impactful projects."},
    {"text":"Q: What is AWS Lambda?\nA: AWS Lambda is a serverless computing service."},
{"text":"Q: What is Amazon RDS?\nA: Amazon RDS is a managed relational database service."},
{"text":"Q: What is Amazon DynamoDB?\nA: DynamoDB is a fully managed NoSQL database service."},
{"text":"Q: What is Amazon VPC?\nA: Amazon VPC allows users to create isolated virtual networks."},
{"text":"Q: What is Elastic Load Balancer?\nA: ELB distributes traffic across multiple resources."},
{"text":"Q: What is Auto Scaling?\nA: Auto Scaling automatically adjusts resources based on demand."},
{"text":"Q: What is CloudWatch?\nA: CloudWatch monitors AWS resources and applications."},
{"text":"Q: What is IAM?\nA: IAM manages access and permissions for AWS resources."},
{"text":"Q: What is Route 53?\nA: Route 53 is AWS's scalable DNS service."},
{"text":"Q: What is AWS CloudFormation?\nA: CloudFormation automates infrastructure deployment using templates."},
    {"text":"Q: What is a binary tree?\nA: A binary tree is a hierarchical data structure where each node has at most two children."},
{"text":"Q: What is a binary search tree?\nA: A BST is a binary tree where left child values are smaller and right child values are larger."},
{"text":"Q: What is DFS?\nA: DFS stands for Depth First Search and explores graph nodes deeply before backtracking."},
{"text":"Q: What is BFS?\nA: BFS stands for Breadth First Search and explores nodes level by level."},
{"text":"Q: What is hashing?\nA: Hashing maps data into fixed-size values for efficient retrieval."},
{"text":"Q: What is dynamic programming?\nA: Dynamic programming solves problems by storing results of subproblems."},
{"text":"Q: What is greedy algorithm?\nA: A greedy algorithm makes the best local choice at each step."},
{"text":"Q: What is merge sort?\nA: Merge sort is a divide and conquer sorting algorithm."},
{"text":"Q: What is quick sort?\nA: Quick sort sorts data using a pivot element."},
{"text":"Q: What is binary search?\nA: Binary search finds elements efficiently in sorted arrays."}
]

dataset = Dataset.from_list(data)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 90
})


In [42]:
# step 4: tokenization
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True,padding="max_length",
    max_length=128)


tokenized_dataset = dataset.map(tokenize_function)

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

In [43]:
from transformers import Trainer , TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_dataset=tokenized_dataset.map(add_labels_to_dataset,batched=True)

# Defensive check and re-application of PEFT if model is not recognized as a PeftModel
# This ensures that the model is correctly prepared for PEFT training
if not isinstance(model, PeftModel):
    print("Warning: Model not recognized as a PEFT model. Attempting to re-apply PEFT configuration.")
    # Redefine lora_config to ensure it's available in this scope
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.5,
        bias="none",
        task_type="CAUSAL_LM"
    )
    # The 'model' variable is assumed to be the base quantized model at this point if it's not a PeftModel.
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)
    print("PEFT configuration re-applied.")

training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

Map:   0%|          | 0/90 [00:00<?, ? examples/s]

Step,Training Loss
10,2.757795
20,1.261216
30,0.688529
40,0.505135
50,0.401317
60,0.381511
70,0.360782
80,0.344644
90,0.324822
100,0.326241


TrainOutput(global_step=115, training_loss=0.6814935756766278, metrics={'train_runtime': 200.5443, 'train_samples_per_second': 2.244, 'train_steps_per_second': 0.573, 'total_flos': 357916763750400.0, 'train_loss': 0.6814935756766278, 'epoch': 5.0})

In [44]:
def generate_response(question):
    prompt = f"Q: {question}\nA:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [45]:
print(generate_response("What is GitHub?"))
print(generate_response("What is SQL?"))
print(generate_response("What is AWS?"))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is GitHub?
A: GitHub.


















































[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is SQL?
A: SQL Server 















































Q: What is AWS?
A: AWS system.















































